In [1]:
# Import necessary libraries
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings(action="ignore")

### Preview the Original Dataset

In [2]:
# Utilize the Pandas library to create a dataframe (df) to visualize our data at a high level
df = pd.read_csv('CC GENERAL.csv')

# Preview the dataframe (displaying all columns)
pd.set_option("display.max_columns", None)
df.head()

,CUST_ID,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
0,C10001,40.900749,0.818182,95.40,0.00,95.4,0.000000,0.166667,0.000000,0.083333,0.000000,0,2,1000.0,201.802084,139.509787,0.000000,12
1,C10002,3202.467416,0.909091,0.00,0.00,0.0,6442.945483,0.000000,0.000000,0.000000,0.250000,4,0,7000.0,4103.032597,1072.340217,0.222222,12
2,C10003,2495.148862,1.000000,773.17,773.17,0.0,0.000000,1.000000,1.000000,0.000000,0.000000,0,12,7500.0,622.066742,627.284787,0.000000,12
3,C10004,1666.670542,0.636364,1499.00,1499.00,0.0,205.788017,0.083333,0.083333,0.000000,0.083333,1,1,7500.0,0.000000,NaN,0.000000,12
4,C10005,817.714335,1.000000,16.00,16.00,0.0,0.000000,0.083333,0.083333,0.000000,0.000000,0,1,1200.0,678.334763,244.791237,0.000000,12


In [3]:
# Review dataset column data types, value counts and total rows/columns
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8950 entries, 0 to 8949
Data columns (total 18 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   CUST_ID                           8950 non-null   object 
 1   BALANCE                           8950 non-null   float64
 2   BALANCE_FREQUENCY                 8950 non-null   float64
 3   PURCHASES                         8950 non-null   float64
 4   ONEOFF_PURCHASES                  8950 non-null   float64
 5   INSTALLMENTS_PURCHASES            8950 non-null   float64
 6   CASH_ADVANCE                      8950 non-null   float64
 7   PURCHASES_FREQUENCY               8950 non-null   float64
 8   ONEOFF_PURCHASES_FREQUENCY        8950 non-null   float64
 9   PURCHASES_INSTALLMENTS_FREQUENCY  8950 non-null   float64
 10  CASH_ADVANCE_FREQUENCY            8950 non-null   float64
 11  CASH_ADVANCE_TRX                  8950 non-null   int64  
 12  PURCHA

## Data Preparation
### Relabel & Save Original Dataset

In [5]:
# Update column labels for better understanding and consistency
# These labels correspond to the data dictionary outlined in the project proposal 
updated_columns = [
    "customer_id",
    "balance",
    "balance_update_frequency",
    "purchases",
    "oneoff_purchases",
    "purchase_installments",
    "cash_advance",
    "purchase_frequency",
    "oneoff_purchase_frequency",
    "purchase_installments_frequency",
    "cash_advance_frequency",
    "cash_advance_transactions",
    "purchase_transactions",
    "credit_limit",
    "payments",
    "minimum_payments",
    "percentage_full_payment",
    "tenure"  
]

relabelled_df = df.set_axis(updated_columns, axis=1)
relabelled_df.head()

,customer_id,balance,balance_update_frequency,purchases,oneoff_purchases,purchase_installments,cash_advance,purchase_frequency,oneoff_purchase_frequency,purchase_installments_frequency,cash_advance_frequency,cash_advance_transactions,purchase_transactions,credit_limit,payments,minimum_payments,percentage_full_payment,tenure
0,C10001,40.900749,0.818182,95.40,0.00,95.4,0.000000,0.166667,0.000000,0.083333,0.000000,0,2,1000.0,201.802084,139.509787,0.000000,12
1,C10002,3202.467416,0.909091,0.00,0.00,0.0,6442.945483,0.000000,0.000000,0.000000,0.250000,4,0,7000.0,4103.032597,1072.340217,0.222222,12
2,C10003,2495.148862,1.000000,773.17,773.17,0.0,0.000000,1.000000,1.000000,0.000000,0.000000,0,12,7500.0,622.066742,627.284787,0.000000,12
3,C10004,1666.670542,0.636364,1499.00,1499.00,0.0,205.788017,0.083333,0.083333,0.000000,0.083333,1,1,7500.0,0.000000,NaN,0.000000,12
4,C10005,817.714335,1.000000,16.00,16.00,0.0,0.000000,0.083333,0.083333,0.000000,0.000000,0,1,1200.0,678.334763,244.791237,0.000000,12


In [8]:
# Save relabelled dataset
relabelled_df.to_csv('cc_dataset.csv', index=False)

In [6]:
# Review dataset column data types, value counts and total rows/columns
relabelled_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8950 entries, 0 to 8949
Data columns (total 18 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   customer_id                      8950 non-null   object 
 1   balance                          8950 non-null   float64
 2   balance_update_frequency         8950 non-null   float64
 3   purchases                        8950 non-null   float64
 4   oneoff_purchases                 8950 non-null   float64
 5   purchase_installments            8950 non-null   float64
 6   cash_advance                     8950 non-null   float64
 7   purchase_frequency               8950 non-null   float64
 8   oneoff_purchase_frequency        8950 non-null   float64
 9   purchase_installments_frequency  8950 non-null   float64
 10  cash_advance_frequency           8950 non-null   float64
 11  cash_advance_transactions        8950 non-null   int64  
 12  purchase_transaction

### Detect & Handle Missing Values

In [7]:
# Check for missing values in our dataset
relabelled_df.isnull().sum()

customer_id                          0
balance                              0
balance_update_frequency             0
purchases                            0
oneoff_purchases                     0
purchase_installments                0
cash_advance                         0
purchase_frequency                   0
oneoff_purchase_frequency            0
purchase_installments_frequency      0
cash_advance_frequency               0
cash_advance_transactions            0
purchase_transactions                0
credit_limit                         1
payments                             0
minimum_payments                   313
percentage_full_payment              0
tenure                               0
dtype: int64

## Exploratory Data Analysis